In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/employee_attrition_processed.csv")
df.head()

,Age,Gender,Marital Status,Department,Job Role,Job Level,Monthly Income,Hourly Rate,Years At Company,Years In Current Role,...,Overtime,Project Count,Average Hours Worked Per Week,Absenteeism,Work Environment Satisfaction,Relationship With Manager,Job Involvement,Distance From Home,Number Of Companies Worked,Attrition
0,58,Female,Married,IT,Manager,1,15488,28,15,4,...,No,6,54,17,4,4,4,20,3,No
1,48,Female,Married,Sales,Assistant,5,13079,28,6,9,...,Yes,2,45,1,4,1,2,25,2,No
2,34,Male,Married,Marketing,Assistant,1,13744,24,24,14,...,Yes,6,34,2,3,4,4,45,3,No
3,27,Female,Divorced,Marketing,Manager,1,6809,26,10,8,...,No,9,48,18,2,3,1,35,3,No
4,40,Male,Divorced,Marketing,Executive,1,10206,52,29,10,...,No,3,33,0,4,1,3,44,3,No


In [ ]:
from sklearn.model_selection import train_test_split

# setting up data
X = df.drop("Attrition", axis=1)
y = df.Attrition.map({"Yes": 1, "No": 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# encoding
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numFeatures = ['Age', 'Job Level', 'Monthly Income', 'Hourly Rate', 'Years At Company',
       'Years In Current Role', 'Years Since Last Promotion',
       'Work Life Balance', 'Job Satisfaction', 'Performance Rating',
       'Training Hours Last Year', 'Project Count',
       'Average Hours Worked Per Week', 'Absenteeism',
       'Work Environment Satisfaction', 'Relationship With Manager',
       'Job Involvement', 'Distance From Home', 'Number Of Companies Worked']
catFeatures = [col for col in X.columns if col not in numFeatures]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numFeatures),
        ("cat", OneHotEncoder(handle_unknown="ignore"), catFeatures)
    ]
)

In [4]:
# model selection
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight="balanced"),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced"),
    'SVM': SVC(probability=True, class_weight="balanced")
}

In [5]:
# logistic regression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

logistic_pipeline = Pipeline([
 ('preprocessor', preprocessor),
 ('classifier', models["Logistic Regression"])
])

logistic_pipeline.fit(X_train, y_train)

# evaluation
log_y_pred = logistic_pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, log_y_pred)}')
print(f'Precision: {precision_score(y_test, log_y_pred)}')
print(f'Recall: {recall_score(y_test, log_y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, log_y_pred)}')
print(f'Confusion matrix: {confusion_matrix(y_test, log_y_pred)}')

Accuracy: 0.57
Precision: 0.14285714285714285
Recall: 0.3548387096774194
ROC AUC: 0.48215308264936063
Confusion matrix: [[103  66]
 [ 20  11]]


In [6]:
# svm
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline([
 ('preprocessor', preprocessor),
 ('classifier', models["SVM"])
])

svm_pipeline.fit(X_train, y_train)

svm_y_pred = svm_pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, svm_y_pred)}')
print(f'Precision: {precision_score(y_test, svm_y_pred)}')
print(f'Recall: {recall_score(y_test, svm_y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, svm_y_pred)}')
print(f'Confusion matrix: {confusion_matrix(y_test, svm_y_pred)}')

Accuracy: 0.66
Precision: 0.16363636363636364
Recall: 0.2903225806451613
ROC AUC: 0.5090666157663676
Confusion matrix: [[123  46]
 [ 22   9]]


In [7]:
# random forest classifier
from sklearn.pipeline import Pipeline

random_forest_pipeline = Pipeline([
 ('preprocessor', preprocessor),
 ('classifier', models["Random Forest"])
])

random_forest_pipeline.fit(X_train, y_train)

rand_forest_y_pred = random_forest_pipeline.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, rand_forest_y_pred)}')
print(f'Precision: {precision_score(y_test, rand_forest_y_pred, zero_division=0)}')
print(f'Recall: {recall_score(y_test, rand_forest_y_pred)}')
print(f'ROC AUC: {roc_auc_score(y_test, rand_forest_y_pred)}')
print(f'Confusion matrix: {confusion_matrix(y_test, rand_forest_y_pred)}')

Accuracy: 0.845
Precision: 0.0
Recall: 0.0
ROC AUC: 0.5
Confusion matrix: [[169   0]
 [ 31   0]]


In [8]:
# saving best model
import joblib
joblib.dump(logistic_pipeline, "../outputs/models/logistic_pipeline.pkl")

['../outputs/models/logistic_pipeline.pkl']

## Machine Learning Conclusions

None of the models performed well during training and they don’t show signs to learn data from the current dataset.

Although random forest achieved the highest accuracy score (0.845), this result is misleading due to the imbalanced nature of the dataset. The confusion matrix showed the model predicted only the majority class (“No Attrition”), resulting in:
-	169 true negatives
-	31 false negatives
-	0 true positives

This demonstrates that accuracy alone isn’t sufficient for evaluating classification models on imbalanced datasets.

Logistic Regression and Support Vector Machine performed slightly better at identifying attrition cases, but both suffered from:
-	Low precision, indicating many false positive predictions
-	Low recall, indicating difficulty identifying actual attrition cases
Additionally, ROC AUC scores close to 0.50 suggest that the models performed only marginally better than random guessing. 

Overall, the results indicate that the current dataset may lack strong predictive signals for attrition, or that employee turnover is influenced by more complex factors not captured by the available features.